In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# --- CONFIGURATION ---
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# --- 1. DATA LOADING ---
print("Fetching Spotify dataset...")
url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-01-21/spotify_songs.csv"
df = pd.read_csv(url)

# --- 2. DATA PREPROCESSING ---
print("Preprocessing data...")
features = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
target = 'playlist_genre'

# Focus on 3 distinct macro-genres to ensure high-quality acoustic separation
selected_genres = ['rock', 'latin', 'edm']

df_clean = df[df[target].isin(selected_genres)].copy()
df_clean = df_clean[features + [target]].dropna()

# Standardize labels for visualization
df_clean[target] = df_clean[target].replace({'latin': 'Latin/Reggaeton', 'edm': 'Electronic', 'rock': 'Rock'})
print(f"Dataset optimized: {df_clean.shape[0]} valid tracks isolated.")

# Split Features and Target
X = df_clean[features]
y = df_clean[target]

# 80% Training, 20% Testing split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scaling (Crucial for distance-based algorithms like KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- 3. MODEL TRAINING ---
print("\nTraining K-Nearest Neighbors (KNN) Model...")
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)
y_pred_knn = knn_model.predict(X_test_scaled)

print("Training Random Forest Classifier Model...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)

# --- 4. EVALUATION & VISUALIZATION ---
def plot_confusion_matrix(y_true, y_pred, title):
    """Generates a heatmap for the confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=sorted(df_clean[target].unique()),
                yticklabels=sorted(df_clean[target].unique()))
    plt.title(title, pad=15, fontsize=14, fontweight='bold')
    plt.ylabel('True Genre', fontweight='bold')
    plt.xlabel('Predicted Genre', fontweight='bold')
    plt.show()

print("\n" + "="*40)
print(f"KNN Accuracy Score: {accuracy_score(y_test, y_pred_knn) * 100:.2f}%")
print("="*40)
plot_confusion_matrix(y_test, y_pred_knn, "Confusion Matrix: K-Nearest Neighbors")

print("\n" + "="*40)
print(f"Random Forest Accuracy Score: {accuracy_score(y_test, y_pred_rf) * 100:.2f}%")
print("="*40)
plot_confusion_matrix(y_test, y_pred_rf, "Confusion Matrix: Random Forest")

# --- 5. FEATURE IMPORTANCE ANALYSIS ---
print("\nGenerating Feature Importance Plot...")
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.title("Audio Features Impact on Genre Prediction (Random Forest)", fontsize=14, fontweight='bold')
plt.bar(range(X.shape[1]), importances[indices], align="center", color='steelblue')
plt.xticks(range(X.shape[1]), [features[i] for i in indices], rotation=45, ha='right')
plt.ylabel('Relative Importance')
plt.tight_layout()
plt.show()